# Agent Function Tools — `@azure/ai-projects`

Demonstrates how to create an agent with function tools, handle function calls, and provide function results to get the final response.

It mirrors the [`agentFunctionTool.ts`](./agentFunctionTool.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
console.log(`Model: ${deploymentName}`);

/**
 * Define a function tool for the model to use
 */
const funcTool = {
  type: "function" as const,
  name: "get_horoscope",
  description: "Get today's horoscope for an astrological sign.",
  strict: true,
  parameters: {
    type: "object",
    properties: {
      sign: {
        type: "string",
        description: "An astrological sign like Taurus or Aquarius",
      },
    },
    required: ["sign"],
    additionalProperties: false,
  },
};

/**
 * Generate a horoscope for the given astrological sign.
 */
function getHoroscope(sign: string): string {
  return `${sign}: Next Tuesday you will befriend a baby otter.`;
}

Model: gpt-5.2


In [2]:
// Create the AI Project client and the OpenAI client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient();

In [3]:
// Create agent with function tools
console.log("Creating agent with function tools...");
const agent = await project.agents.createVersion("function-tool-agent", {
  kind: "prompt",
  model: deploymentName,
  instructions: "You are a helpful assistant that can use function tools.",
  tools: [funcTool],
});
console.log(`Agent created (id: ${agent.id}, name: ${agent.name}, version: ${agent.version})`);

Creating agent with function tools...
Agent created (id: function-tool-agent:1, name: function-tool-agent, version: 1)


In [4]:
// Prompt the model with tools defined
console.log("\nGenerating initial response...");
const response = await openAIClient.responses.create(
  {
    input: [
      {
        type: "message",
        role: "user",
        content: "What is my horoscope? I am an Aquarius.",
      },
    ],
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);
console.log(`Response output: ${JSON.stringify(response.output, null, 2)}`);


Generating initial response...
Response output: [
  {
    "type": "function_call",
    "id": "fc_0e0d2bad905095b9006a5fdd3205248196aefc19dcfc21f201",
    "agent_reference": {
      "type": "agent_reference",
      "name": "function-tool-agent",
      "version": "1"
    },
    "response_id": "resp_0e0d2bad905095b9006a5fdd3173f0819683b1ed8beb054c26",
    "call_id": "call_JR4ViyQ1PlKL0uD6V0iHtfd0",
    "name": "get_horoscope",
    "arguments": "{\"sign\":\"Aquarius\"}",
    "status": "completed"
  }
]


In [5]:
// Process function calls
const inputList: Array<{
  type: "function_call_output";
  call_id: string;
  output: string;
}> = [];

for (const item of response.output) {
  if (item.type === "function_call") {
    if (item.name === "get_horoscope") {
      // Parse the function arguments
      const args = JSON.parse(item.arguments);

      // Execute the function logic for get_horoscope
      const horoscope = getHoroscope(args.sign);

      // Provide function call results to the model
      inputList.push({
        type: "function_call_output",
        call_id: item.call_id,
        output: JSON.stringify({ horoscope }),
      });
    }
  }
}

console.log("\nFinal input:");
console.log(JSON.stringify(inputList, null, 2));


Final input:
[
  {
    "type": "function_call_output",
    "call_id": "call_JR4ViyQ1PlKL0uD6V0iHtfd0",
    "output": "{\"horoscope\":\"Aquarius: Next Tuesday you will befriend a baby otter.\"}"
  }
]
[
  {
    "type": "function_call_output",
    "call_id": "call_JR4ViyQ1PlKL0uD6V0iHtfd0",
    "output": "{\"horoscope\":\"Aquarius: Next Tuesday you will befriend a baby otter.\"}"
  }
]


In [6]:
// Submit function results to get final response
const finalResponse = await openAIClient.responses.create(
  {
    input: inputList,
    previous_response_id: response.id,
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);

// The model should be able to give a response!
console.log("\nFinal output:");
console.log(JSON.stringify(finalResponse, null, 2));


Final output:
{
  "metadata": {},
  "top_logprobs": 0,
  "temperature": 1,
  "top_p": 0.98,
  "service_tier": "default",
  "prompt_cache_retention": "in_memory",
  "previous_response_id": "resp_0e0d2bad905095b9006a5fdd3173f0819683b1ed8beb054c26",
  "model": "gpt-5.2",
  "reasoning": {
    "effort": "none",
    "context": "current_turn",
    "mode": "standard"
  },
  "background": false,
  "text": {
    "format": {
      "type": "text"
    },
    "verbosity": "medium"
  },
  "tools": [
    {
      "type": "function",
      "name": "get_horoscope",
      "description": "Get today's horoscope for an astrological sign.",
      "parameters": {
        "type": "object",
        "properties": {
          "sign": {
            "type": "string",
            "description": "An astrological sign like Taurus or Aquarius"
          }
        },
        "required": [
          "sign"
        ],
        "additionalProperties": false
      },
      "strict": true
    }
  ],
  "tool_choice": "auto",
 

In [7]:
// Clean up
console.log("\nCleaning up resources...");
await project.agents.deleteVersion(agent.name, agent.version);
console.log("Agent deleted");


Cleaning up resources...
Agent deleted
